# Composite phase fractions from CT: full pipeline on Google Colab

Builds the synthetic dataset, trains the 3D ResNet, and reproduces every result in the paper.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (any GPU works).
2. *Runtime → Run all*. The code is fetched from GitHub into `MyDrive/project65/p65` the first time.

**If Colab disconnects:** reconnect and run all cells again from the top. Dataset building and
training resume from where they stopped (data, checkpoints and results live in Drive).

| Section | What it does | Time on a T4 |
|---|---|---|
| 1-2 | Setup, ground-truth tests, dataset (66 composites x 3 doses) | ~40 min |
| 3-4 | Training (3 seeds), leakage control, Experiment 1 | ~45 min |
| 5 | Experiments 2-5 (crop size, void level, packing, dose) | ~10 min |


## 0. Settings
Change only these. `SMOKE = True` runs a 5-minute miniature of the whole pipeline first.

In [ ]:
import os
SMOKE = os.environ.get("P65_SMOKE", "0") == "1"   # set True to test the pipeline end to end quickly

N_PARENTS, N_OOD, SIZE, ANGLES = (60, 6, 128, 360) if not SMOKE else (8, 2, 64, 120)
EPOCHS, STEPS, CROP, BS = (60, 400, 64, 16) if not SMOKE else (2, 32, 32, 8)
SHUFFLE_EPOCHS = 10 if not SMOKE else 1
EVAL_CROPS = [32, 64, 96] if not SMOKE else [16, 32, 48]
print("SMOKE mode" if SMOKE else "FULL mode", N_PARENTS, SIZE, EPOCHS)

## 1. Setup: GPU, Google Drive, code, packages

In [ ]:
import subprocess, sys
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout or "no GPU")
except FileNotFoundError:
    print("No GPU found. In Colab: Runtime -> Change runtime type -> T4 GPU, then rerun.")

In [ ]:
import os, shutil, zipfile, pathlib, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content/drive/MyDrive/project65")   # persistent: survives disconnects
    LOCAL = pathlib.Path("/content")                           # fast local disk, wiped on disconnect
except ImportError:                                            # (only for testing outside Colab)
    ROOT = pathlib.Path(os.environ["P65_ROOT"]); LOCAL = pathlib.Path(os.environ["P65_LOCAL"])

ROOT.mkdir(parents=True, exist_ok=True)
REPO_URL = "https://github.com/Abhnvsingh/composite-ct-phase-fractions.git"
CODE = ROOT / "p65" / "src"
if (ROOT / "p65" / ".git").exists():
    subprocess.run(["git", "-C", str(ROOT / "p65"), "pull", "-q"], check=False)   # get the latest code
elif (ROOT / "project65_starter_code.zip").exists():
    zipfile.ZipFile(ROOT / "project65_starter_code.zip").extractall(ROOT)        # offline alternative
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT / "p65")], check=True)
tag = "smoke" if SMOKE else "full"
DATA_DRIVE = ROOT / f"data_{tag}"          # dataset (persistent)
DATA = LOCAL / f"data_{tag}"               # local copy for fast training
RUNS = ROOT / f"runs_{tag}"                # checkpoints + results (persistent)
os.chdir(CODE)

def run(cmd):
    # Run a script, show its output live, and STOP the notebook if it fails
    # (a plain !python line fails silently and a later cell then shows a confusing error).
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}); the error is printed above.")

print("code:", CODE, "\ndata:", DATA_DRIVE, "\nruns:", RUNS)

In [ ]:
!pip -q install astra-toolbox
import sys; sys.path.insert(0, str(CODE))
import astra, torch
from ct_sim import gpu_available
print("torch", torch.__version__, "| CUDA for training:", torch.cuda.is_available())
print("astra", astra.__version__, "| GPU CT simulation:", gpu_available())

## 2. check the ground truth, then build the dataset
`tests.py` must print **ALL PASS** before anything else is trusted.

In [ ]:
run("python tests.py")

Full build: about 66 parents × 3 doses. On a 2-core CPU it took 63 s per parent;
on a Colab GPU the CT simulation runs on the GPU, so expect it to be faster (watch the
per-parent seconds it prints). If Colab disconnects, rerun this cell: it resumes.

In [ ]:
run(f"python make_dataset.py --n {N_PARENTS} --n_ood {N_OOD} --size {SIZE} --angles {ANGLES} --out {DATA_DRIVE}")

In [ ]:
import pandas as pd, json
parents = pd.read_csv(DATA_DRIVE / "parents.csv")
print(json.loads((DATA_DRIVE / "config.json").read_text())["device"], "simulation |",
      len(parents), "parents |", parents.split.value_counts().to_dict())
parents[["true_fibre", "true_void", "r_mean", "cnr_high", "cnr_medium", "cnr_low"]].describe().round(3)

Copy the dataset from Drive to Colab's local disk; training reads it thousands of times.

In [ ]:
if not (DATA / "parents.csv").exists() or len(pd.read_csv(DATA / "parents.csv")) != len(parents):
    shutil.copytree(DATA_DRIVE, DATA, dirs_exist_ok=True)
print(sum(f.stat().st_size for f in DATA.glob("*")) / 1e9, "GB in", DATA)

Look at one slice (your realism check, and a figure for the report).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
pid = int(parents[parents.split == "test"].id.iloc[0]); z = SIZE // 2
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(np.load(DATA / f"p{pid:03d}_labels.npy")[z], cmap="viridis"); ax[0].set_title("labels")
for a_, d in zip(ax[1:], ["high", "medium", "low"]):
    a_.imshow(np.load(DATA / f"p{pid:03d}_rec_{d}.npy")[z].astype(float), cmap="gray", vmin=0.3, vmax=1.2)
    a_.set_title(f"{d} dose, CNR {parents.set_index('id').loc[pid, 'cnr_' + d]:.1f}")
for a_ in ax: a_.axis("off")
plt.show()

## 3. train the 3D ResNet
Checkpoints are saved to Drive every epoch. After a disconnect, rerun from the top and this cell
continues from the last epoch. After 2 epochs, check the seconds per epoch: if
`seconds × EPOCHS` is more than ~4 h, lower `EPOCHS` in Settings and add `--fresh`.

In [ ]:
run(f"python train.py --data {DATA} --out {RUNS}/c{CROP} --crop {CROP} --epochs {EPOCHS} --steps {STEPS} --bs {BS} --workers 2")

In [ ]:
log = pd.read_csv(RUNS / f"c{CROP}" / "log.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log.epoch, log.train_loss); ax[0].set_yscale("log"); ax[0].set_title("training loss")
for c in ["val_mae_fibre", "val_mae_matrix", "val_mae_void"]:
    ax[1].plot(log.epoch, log[c], label=c.replace("val_mae_", ""))
ax[1].set_title("validation MAE (percentage points)"); ax[1].legend(); plt.show()
print(f"{log.sec.median():.0f} s/epoch; best epoch {log.val_mae_fibre.idxmin()}")

**Leakage test.** Trained on shuffled targets, validation MAE must stay high (near the predict-the-mean level). If it gets as good as the real model, something leaks.

In [ ]:
run(f"python train.py --data {DATA} --out {RUNS}/shuffled --crop {CROP} --epochs {SHUFFLE_EPOCHS} --steps {STEPS} --bs {BS} --workers 2 --shuffle_labels")
real = pd.read_csv(RUNS / f"c{CROP}" / "log.csv"); shuf = pd.read_csv(RUNS / "shuffled" / "log.csv")
print("best val fibre MAE  real:", round(real.val_mae_fibre.min(), 2), " shuffled:", round(shuf.val_mae_fibre.min(), 2))

## 4. Experiment 1: CNN vs baselines on the test set, plus out-of-distribution

In [ ]:
CK = RUNS / f"c{CROP}" / "best.pt"
for split in ["test", "ood"]:
    if not (RUNS / f"c{CROP}" / f"eval_{split}_c{CROP}" / "metrics.csv").exists():
        run(f"python evaluate.py --data {DATA} --ckpt {CK} --crop {CROP} --split {split}")

In [ ]:
from IPython.display import Image, display
E = RUNS / f"c{CROP}" / f"eval_test_c{CROP}"
m = pd.read_csv(E / "metrics.csv")
display(m[m.dose == "all"].round(3))
for ph in ["fibre", "void"]:
    display(Image(str(E / f"scatter_{ph}.png"), width=380))
    display(Image(str(E / f"bland_altman_{ph}_cnn_vs_truth.png"), width=420))
o = pd.read_csv(RUNS / f"c{CROP}" / f"eval_ood_c{CROP}" / "metrics.csv")
print("OOD (fibre radius never seen in training):"); display(o[o.dose == "all"].round(3))

## 5. Experiments 2-5
Same trained model, no retraining: evaluate at three crop sizes, then bin by void, packing and dose.

In [ ]:
for c in EVAL_CROPS:
    if not (RUNS / f"c{CROP}" / f"eval_test_c{c}" / "metrics.csv").exists():
        run(f"python evaluate.py --data {DATA} --ckpt {CK} --crop {c} --split test")
evals = " ".join(str(RUNS / f"c{CROP}" / f"eval_test_c{c}") for c in EVAL_CROPS)
EXP = RUNS / f"c{CROP}" / "experiments"
run(f"python experiments.py --data {DATA} --evals {evals} --main {RUNS}/c{CROP}/eval_test_c{CROP} --out {EXP}")

In [ ]:
for f in ["exp2_size", "exp3_void", "exp4_packing", "exp5_noise"]:
    print(f); display(Image(str(EXP / f"{f}.png"), width=900))

## 6. Strengthen Experiment 1 (what a reviewer will ask for)
1. **Two more training seeds** (same settings, ~10 min each): is the result stable?
2. **A fair classical baseline:** multi-Otsu after Gaussian smoothing, on the identical test crops.
3. **Paired significance:** per test volume, CNN error vs each baseline, Wilcoxon signed-rank test.
Already-finished seeds are skipped on rerun.

In [ ]:
SEEDS = [1, 2] if not SMOKE else [1]
seed_evals = [RUNS / f"c{CROP}" / f"eval_test_c{CROP}"]
for s in SEEDS:
    sd = RUNS / f"c{CROP}_seed{s}"
    run(f"python train.py --data {DATA} --out {sd} --crop {CROP} --epochs {EPOCHS} --steps {STEPS} --bs {BS} --workers 2 --seed {s}")
    if not (sd / f"eval_test_c{CROP}" / "metrics.csv").exists():
        run(f"python evaluate.py --data {DATA} --ckpt {sd}/best.pt --crop {CROP} --split test")
    seed_evals.append(sd / f"eval_test_c{CROP}")
STATS = RUNS / f"c{CROP}" / "stats"
run(f"python stats.py --data {DATA} --eval {seed_evals[0]} --out {STATS} --seed_evals " + " ".join(map(str, seed_evals)))

In [ ]:
for f in ["exp1_with_fair_baseline", "paired_tests", "seeds_summary"]:
    print(f); display(pd.read_csv(STATS / f"{f}.csv").round(4))

## 7. Download all results
Tables and figures for the report (model weights stay in Drive).

In [ ]:
out_zip = shutil.make_archive(str(LOCAL / f"project65_results_{tag}"), "zip", RUNS)
print(out_zip)
try:
    from google.colab import files; files.download(out_zip)
except ImportError:
    pass